## Perfil de qualidade do Bronze

Antes de transformar, verifico os dados brutos atributo por atributo nas
5 dimensões de qualidade: Completude, Unicidade, Consistência, Acurácia
e Outliers. O perfil cobre a base inteira, não só o período 2021–2025.
O que for encontrado aqui é tratado nas etapas seguintes deste notebook.

In [0]:
# Leitura do Dataset

from pyspark.sql import functions as F

BASE = "/Volumes/susep_capitalizacao/bronze/raw_susep_capitalizacao"

def ler(arquivo, encoding):
    return spark.read.csv(f"{BASE}/{arquivo}", header=True, sep=";", encoding=encoding)

bronze = {
    "ses_cap_uf":    ler("ses_cap_uf.csv", "ASCII"),
    "Ses_Dados_Cap": ler("Ses_Dados_Cap.csv", "ISO-8859-1"),
    "Ses_prov":      ler("Ses_prov.csv", "ASCII"),
    "Ses_cias":      ler("Ses_cias.csv", "ISO-8859-1"),
}

In [0]:
# Completude (nulos ou vazios por coluna)

linhas = []
for nome, df in bronze.items():
    total = df.count()
    vazios = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).first().asDict()
    for c, n in vazios.items():
        linhas.append((nome, c, total, n, round(100 * n / total, 2)))

display(spark.createDataFrame(
    linhas, "arquivo string, coluna string, total_linhas long, nulos_ou_vazios long, pct double"))

In [0]:
# Unicidade (chaves duplicadas)

chaves = {
    "ses_cap_uf":    ["COENTI", "DAMESANO", "UF"],
    "Ses_Dados_Cap": ["coenti", "damesano", "codModal"],
    "Ses_prov":      ["coenti", "damesano"],
    "Ses_cias":      ["Coenti"],
}

linhas = []
for nome, cols in chaves.items():
    df = bronze[nome]
    k = [F.upper(F.trim(F.col(c))).alias(c) for c in cols]
    total          = df.count()
    chaves_unicas  = df.select(k).distinct().count()
    dup_chave      = total - chaves_unicas
    # Só calcula o distinct completo se há duplicata por chave:
    # sem duplicata por chave não pode haver duplicata exata.
    linhas_unicas  = df.distinct().count() if dup_chave > 0 else total
    dup_exatas     = total - linhas_unicas
    linhas.append((nome, ", ".join(cols), total, dup_chave, dup_exatas, dup_chave - dup_exatas))

resumo = spark.createDataFrame(linhas,
    "arquivo string, chave string, linhas long, dup_por_chave long, "
    "dup_exatas long, dup_conflitantes long")
display(resumo)

# Detalhe: para cada arquivo com duplicata, mostra as chaves repetidas
for r in resumo.filter("dup_por_chave > 0").collect():
    cols = chaves[r.arquivo]
    print(f"Chaves duplicadas em {r.arquivo}:")
    display(bronze[r.arquivo]
            .groupBy([F.upper(F.trim(F.col(c))).alias(c) for c in cols])
            .count().filter("count > 1"))

In [0]:
# Consistência por coluna (formato)

UFS = ["AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT","PA",
       "PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"]

padroes = {
    "codigo":  "^[0-9]{5}$",
    "mes":     "^[0-9]{4}(0[1-9]|1[0-2])$",
    "decimal": "^-?[0-9]+(,[0-9]+)?$",
    "inteiro": "^-?[0-9]+$",
}

chaves = {
    "ses_cap_uf":    ["COENTI", "DAMESANO", "UF"],
    "Ses_Dados_Cap": ["coenti", "damesano", "codModal"],
    "Ses_prov":      ["coenti", "damesano"],
    "Ses_cias":      ["Coenti"],
}

esquema = {
    "ses_cap_uf": {
        "COENTI": "codigo", "DAMESANO": "mes", "UF": "uf",
        "PREMIO": "decimal", "RESGPAGO": "decimal", "SORTPAGO": "decimal",
        "NUMPARTIC": "decimal", "RESGATANTES": "inteiro", "SORTEIOS": "inteiro",
    },
    "Ses_Dados_Cap": {
        "coenti": "codigo", "damesano": "mes", "codModal": "inteiro", "modalidade": "texto",
        "receitasCap": "decimal", "valorResg": "decimal", "sorteiosPagos": "decimal",
    },
    "Ses_prov": {"coenti": "codigo", "damesano": "mes", "valor": "decimal"},
    "Ses_cias": {
        "Coenti": "codigo", "Noenti": "texto",
        "Cogrupo": "fora_do_modelo", "Nogrupo": "fora_do_modelo",
    },
}

def invalido(c, regra):
    v = F.trim(F.col(c))                      # conteúdo, sem espaços nas pontas
    if regra == "uf":    return ~v.isin(UFS)
    if regra == "texto": return F.lit(False)  # texto livre: sem regra de formato
    return ~v.rlike(padroes[regra])

linhas, exemplos = [], {}
for arq, regras in esquema.items():
    df = bronze[arq]
    sem_regra = set(df.columns) ^ set(regras)
    if sem_regra:
        print(f"ATENÇÃO {arq}: colunas sem regra ou inexistentes -> {sem_regra}")
    for c, regra in regras.items():
        if regra == "fora_do_modelo":
            linhas.append((arq, c, regra, None, None, None))
            continue
        # vazios já medidos na Completude
        testados = df.filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""))
        ruins = testados.filter(invalido(c, regra))
        n_invalidos = ruins.count()
        n_espacos = testados.filter(F.col(c) != F.trim(F.col(c))).count()
        linhas.append((arq, c, regra, testados.count(), n_invalidos, n_espacos))
        if n_invalidos:
            cols = list(dict.fromkeys(chaves[arq] + [c]))   # chave + coluna, sem repetir
            exemplos[(arq, c)] = ruins.select(cols).limit(5)

display(spark.createDataFrame(linhas,
    "arquivo string, coluna string, regra string, valores_testados long, "
    "invalidos long, com_espacos_nas_pontas long"))

for (arq, c), ex in exemplos.items():
    print(f"Exemplos de valor inválido — {arq}.{c}")
    display(ex)

In [0]:
# Consistência entre colunas (integridade referencial)

# Integridade referencial: toda empresa dos arquivos de valores existe no cadastro?
cias = bronze["Ses_cias"].select(F.trim("Coenti").alias("k")).distinct()
ref = []
for arq, c in [("ses_cap_uf", "COENTI"), ("Ses_Dados_Cap", "coenti"), ("Ses_prov", "coenti")]:
    orfas = bronze[arq].select(F.trim(c).alias("k")).join(cias, "k", "left_anti")
    ref.append((arq, c, orfas.count(), orfas.distinct().count()))
display(spark.createDataFrame(ref,
    "arquivo string, coluna string, linhas_sem_cadastro long, empresas_sem_cadastro long"))

# Cada código de modalidade tem uma única descrição?
display(bronze["Ses_Dados_Cap"]
        .groupBy("codModal")
        .agg(F.countDistinct(F.coalesce("modalidade", F.lit(""))).alias("qtd_descricoes"),
             F.collect_set("modalidade").alias("descricoes"))
        .orderBy(F.col("codModal").cast("int")))

In [0]:
for nome, df in bronze.items():
    df.createOrReplaceTempView(nome)

In [0]:
%sql
-- Acurácia (valores negativos)

WITH valores AS (
  SELECT 'ses_cap_uf' AS arquivo, DAMESANO AS damesano, 'PREMIO'      AS coluna, PREMIO      AS valor FROM ses_cap_uf
  UNION ALL SELECT 'ses_cap_uf',    DAMESANO, 'RESGPAGO',    RESGPAGO    FROM ses_cap_uf
  UNION ALL SELECT 'ses_cap_uf',    DAMESANO, 'SORTPAGO',    SORTPAGO    FROM ses_cap_uf
  UNION ALL SELECT 'ses_cap_uf',    DAMESANO, 'NUMPARTIC',   NUMPARTIC   FROM ses_cap_uf
  UNION ALL SELECT 'ses_cap_uf',    DAMESANO, 'RESGATANTES', RESGATANTES FROM ses_cap_uf
  UNION ALL SELECT 'ses_cap_uf',    DAMESANO, 'SORTEIOS',    SORTEIOS    FROM ses_cap_uf
  UNION ALL SELECT 'Ses_Dados_Cap', damesano, 'receitasCap',   receitasCap   FROM Ses_Dados_Cap
  UNION ALL SELECT 'Ses_Dados_Cap', damesano, 'valorResg',     valorResg     FROM Ses_Dados_Cap
  UNION ALL SELECT 'Ses_Dados_Cap', damesano, 'sorteiosPagos', sorteiosPagos FROM Ses_Dados_Cap
  UNION ALL SELECT 'Ses_prov',      damesano, 'valor',         valor         FROM Ses_prov
)
SELECT
  arquivo,
  coluna,
  COUNT_IF(CAST(REPLACE(valor, ',', '.') AS DOUBLE) < 0) AS negativos_total,
  COUNT_IF(CAST(REPLACE(valor, ',', '.') AS DOUBLE) < 0
           AND damesano BETWEEN '202101' AND '202512')   AS negativos_2021_2025
FROM valores
GROUP BY arquivo, coluna
ORDER BY arquivo, coluna

In [0]:
import matplotlib.pyplot as plt

# 1. Prêmio anual por empresa, com nome (SQL simples → pandas)
df = spark.sql("""
    SELECT LEFT(u.DAMESANO, 4)                                          AS ano,
           TRIM(c.Noenti)                                               AS empresa,
           SUM(CAST(REPLACE(u.PREMIO, ',', '.') AS DOUBLE)) / 1e9       AS premio_bi
    FROM ses_cap_uf u
    JOIN Ses_cias c ON TRIM(c.Coenti) = TRIM(u.COENTI)
    WHERE u.DAMESANO BETWEEN '202101' AND '202512'
    GROUP BY 1, 2
""").toPandas()

anos = sorted(df["ano"].unique())

# 2. Boxplot por ano (os pontos fora da caixa são os outliers pela regra de Tukey)
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([df[df["ano"] == a]["premio_bi"] for a in anos])
ax.set_xticks(range(1, len(anos) + 1), anos)

# 3. Nome das empresas acima do limite
for i, a in enumerate(anos, start=1):
    d = df[df["ano"] == a]
    q1, q3 = d["premio_bi"].quantile([0.25, 0.75])
    limite = q3 + 1.5 * (q3 - q1)
    for _, r in d[d["premio_bi"] > limite].iterrows():
        ax.annotate(r["empresa"].split()[0], (i, r["premio_bi"]), xytext=(6, 0), textcoords="offset points", fontsize=8)

ax.set_ylabel("Prêmio anual (R$ bilhões)")
ax.set_title("Prêmio anual por empresa, 2021–2025 (pontos = outliers pela regra de Tukey)")
plt.show()